# Chapter 14: list visuals without a title or an axis unit

Point `REPORT` at a report saved in the Power BI Project (PBIP) format with the enhanced report format (PBIR),
where each visual is a `visual.json` file. The sample report in `../report` is synthetic and hand written,
not exported from Power BI Desktop. Check one of your own exports before you trust the rules below.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd

REPORT = Path("../report/Kestrel Cellar.Report")
UNIT = re.compile(r"\b(hL|kg|USD|ppb|ppm|EBC|IBU|min|minutes|hours|cans|percent|degrees Plato|degrees Celsius)\b|%")
AXIS_VISUALS = {"lineChart", "areaChart", "barChart", "columnChart", "clusteredBarChart", "clusteredColumnChart",
                "stackedBarChart", "stackedColumnChart", "waterfallChart", "scatterChart", "lineClusteredColumnComboChart"}

def literal(objects, key, prop):
    try:
        return str(objects[key][0]["properties"][prop]["expr"]["Literal"]["Value"]).strip("'")
    except (KeyError, IndexError, TypeError):
        return None

In [ ]:
rows = []
for f in sorted(REPORT.rglob("visual.json")):
    v = json.loads(f.read_text())
    vis = v.get("visual", {})
    kind = vis.get("visualType", "unknown")
    box = v.get("visualContainerObjects", {})
    title = literal(box, "title", "text")
    problems = []
    if literal(box, "title", "show") == "false" or not title:
        problems.append("no title")
    if kind in AXIS_VISUALS:
        axis = literal(vis.get("objects", {}), "valueAxis", "titleText")
        if not axis or not UNIT.search(axis):
            problems.append("no unit on the value axis")
    rows.append([f.parent.parent.parent.name, v.get("name"), kind, title or "", "; ".join(problems) or "ok"])
lint = pd.DataFrame(rows, columns=["page", "visual", "type", "title", "finding"])
print(lint.to_string(index=False))
print(f"\n{(lint.finding != 'ok').sum()} of {len(lint)} visuals need a fix")

**What to look at.** Each finding is a missing label, the same thing a brewer asks about in the walkthrough of 14.5.